# COMPSCI5094 Coursework - Conversational Interfaces: Premier League Dialogue Agent
**Author**: Patricia Natasha Munginga

**Student ID**: 3033388M

**Date**: February 2026

**Project**: Task‑Oriented Dialogue System for Premier League Information

This notebook documents the development of a task‑oriented conversational agent that provides information about English Premier League teams and personnel using a football data API and an LLM‑based NLU component.
The focus is on defining intents and slots, implementing robust intent detection and slot filling with a local LLM, integrating real‑time football data via an external API and demonstrating end‑to‑end dialogues that satisfy user queries while enabling clear evaluation of NLU performance.

### Setup and Imports

In [4]:
"""Installs and Imports"""

# Install the Ollama python library
%pip install ollama

# Import the Ollama python library
import ollama

# API functionality
import requests
import json

Note: you may need to restart the kernel to use updated packages.


### Configuration

In [5]:
# Create a client connected to the local Ollama server
client = ollama.Client(host='http://localhost:11434')

# Ensure the required model is available locally (downloads it if needed)
!ollama pull qwen3:4b-instruct

# Define the model name to use throughout the notebook
MODEL = 'qwen3:4b-instruct'  

# Database and API 
FOOTBALL_DATA_API_KEY = "7b428478107f4bdb98af277f262c4350"
FOOTBALL_DATA_BASE_URL = "https://api.football-data.org/v4"
PREMIER_LEAGUE_CODE = "PL" # English Premier League

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 85e4a5b7b8ef: 100% ▕██████████████████▏ 2.5 GB                         
pulling eade0a07cac7: 100% ▕██████████████████▏ 1.4 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling 0914c7781e00: 100% ▕██████████████████▏  119 B                         
pulling b72accf9724e: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


### API Functions

In [ ]:
# Function to call the football API
def call_football_data_api(endpoint, params=None, verbose=True):
    """
    Generic helper function to call the football-data API.

    endpoint: string like 'teams/57'
    params: dictionary of query parameters (e.g., {'status': 'SCHEDULED'})
    verbose: if True, prints basic debug info
    """
    headers = {'X-Auth-Token': FOOTBALL_DATA_API_KEY}
    url = f"{FOOTBALL_DATA_BASE_URL}/{endpoint}"

    try:
        response = requests.get(url, headers=headers, params=params, timeout=10)

        if verbose:
            print(f"[API] GET {url} params={params} status={response.status_code}")

        response.raise_for_status()
        return response.json()

    except requests.exceptions.RequestException as e:
        # Return a structured error (better for tool outputs)
        return {"error": str(e), "endpoint": endpoint, "params": params}
    
# Function to get team information
    